
# Control de parámetros de extracción

## Propósito

Obtener y validar el corte de extracción correspondiente al pipeline antes de iniciar el proceso de ingesta, generando además un identificador único para la ejecución y registrando la trazabilidad del proceso.

## Flujo

1. Leer la tabla Delta de control.

2. Obtener la configuración del pipeline.

3. Validar la existencia de un único corte.

4. Validar la consistencia del rango de fechas.

5. Generar un `id_ejecucion` único para la ejecución completa del Workflow.

6. Generar un `log_id` único para identificar la ejecución de este notebook.

7. Registrar el inicio de la ejecución en `etl_log` con estado `EN_PROCESO`.

8. Publicar `fecha_inicio`, `fecha_fin` e `id_ejecucion` mediante Task Values.

9. Registrar la finalización de la ejecución en `etl_log` con estado `EXITOSO`.

## Tabla de control

`workspace.metadata.pipeline_control`

## Tabla de ejecución

`workspace.metadata.ejecucion_control`

## Tabla de trazabilidad ETL

`workspace.metadata.etl_log`

## Parámetros publicados

* `fecha_inicio`
* `fecha_fin`
* `id_ejecucion`
* `log_id`

Estos valores son utilizados por las siguientes Tasks del Workflow para mantener la trazabilidad de extremo a extremo.

`id_ejecucion` identifica la ejecución completa del Workflow, mientras que `log_id` identifica específicamente la ejecución de este notebook en `etl_log`.

## Control de ejecución

El registro de este notebook en `etl_log` sigue el siguiente flujo:

`EN_PROCESO → EXITOSO`

En caso de error, el mecanismo de manejo de errores del Workflow actualizará el registro correspondiente con estado `ERROR`.

## Dependencias

Este notebook debe ejecutarse como una **Task dentro de un Databricks Workflow** y debe ser la primera etapa del proceso de extracción, ya que publica los parámetros que serán consumidos por las siguientes Tasks mediante Task Values.

## Resultado esperado

Un único corte de extracción validado, un `id_ejecucion` generado para identificar la ejecución completa del pipeline y los parámetros publicados correctamente para las siguientes Tasks del Workflow.


In [0]:
# ============================================================
# GENERACIÓN DE IDs
# ============================================================

import uuid

id_ejecucion = str(uuid.uuid4())
log_id = str(uuid.uuid4())

print(f"ID de ejecución: {id_ejecucion}")
print(f"Log ID: {log_id}")

In [0]:
# ============================================================
# CONFIGURACIÓN DE TABLAS
# ============================================================

# Define la tabla donde se registrará el estado
# de ejecución del proceso ETL.
TABLA_ETL_LOG = "prod_dataco.metadata.etl_log"

In [0]:
# ============================================================
# INICIO DE LOG ETL
# ============================================================

from pyspark.sql import functions as F


df_log_inicio = spark.createDataFrame(
    [
        (
            log_id,
            id_ejecucion,
            "00_prepare_pipeline_control",
            "metadata",
            None,
            None,
            None,
            "EN_PROCESO",
            None
        )
    ],
    """
    log_id STRING,
    id_ejecucion STRING,
    notebook STRING,
    capa STRING,
    fecha_inicio TIMESTAMP,
    fecha_fin TIMESTAMP,
    filas_procesadas BIGINT,
    estado STRING,
    mensaje_error STRING
    """
)

df_log_inicio = (
    df_log_inicio
    .withColumn("fecha_inicio", F.current_timestamp())
)

df_log_inicio.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TABLA_ETL_LOG)

print("✓ ETL Log iniciado")
print(f"✓ log_id: {log_id}")
print(f"✓ id_ejecucion: {id_ejecucion}")
print("✓ Estado: EN_PROCESO")

In [0]:
# ============================================================
# PUBLICACIÓN DE IDs PARA EL WORKFLOW
# ============================================================

dbutils.jobs.taskValues.set(
    key="id_ejecucion",
    value=id_ejecucion
)

dbutils.jobs.taskValues.set(
    key="log_id",
    value=log_id
)

print("✓ IDs publicados correctamente")
print(f"✓ id_ejecucion: {id_ejecucion}")
print(f"✓ log_id: {log_id}")

In [0]:
# ============================================================
# CREACIÓN DEL SCHEMA DE METADATA
# ============================================================

# Crea el schema de metadata si aún no existe en el catálogo prod_dataco.
# Este schema almacenará las tablas de control del pipeline,
# como pipeline_control, ejecucion_control y etl_log.
spark.sql("""
CREATE SCHEMA IF NOT EXISTS prod_dataco.metadata
""")

In [0]:
# ============================================================
# VALIDACIÓN DEL SCHEMA
# ============================================================

# Muestra los schemas existentes dentro del catálogo prod_dataco
# para verificar que prod_dataco.metadata fue creado correctamente.
spark.sql("SHOW SCHEMAS IN prod_dataco").show(truncate=False)

In [0]:
# ============================================================
# CREACIÓN DE TABLA DE CONTROL DEL PIPELINE
# ============================================================

# Crea la tabla pipeline_control en formato Delta dentro de prod_dataco.metadata
# si aún no existe. Esta tabla almacena la configuración y el estado del pipeline,
# incluyendo el rango de fechas que corresponde procesar.

spark.sql("""
CREATE TABLE IF NOT EXISTS prod_dataco.metadata.pipeline_control (
    pipeline_id             INT          COMMENT 'Identificador numérico único del pipeline',
    pipeline_name           STRING       COMMENT 'Nombre descriptivo del pipeline u orquestación',
    task_name               STRING       COMMENT 'Nombre de la tarea específica dentro del pipeline',
    load_type               STRING       COMMENT 'Tipo de carga (ej. FULL, INCREMENTAL)',
    load_frequency          STRING       COMMENT 'Frecuencia de ejecución (ej. DAILY, HOURLY, MONTHLY)',
    last_successful_end_date DATE        COMMENT 'Última fecha de datos procesada con éxito',
    next_start_date         DATE        COMMENT 'Fecha de inicio del siguiente rango a procesar',
    next_end_date           DATE        COMMENT 'Fecha de fin del siguiente rango a procesar',
    status                  STRING       COMMENT 'Estado actual de la regla (ej. ACTIVE, INACTIVE, PAUSED)',
    created_at              TIMESTAMP    COMMENT 'Marca de tiempo de creación del registro',
    updated_at              TIMESTAMP    COMMENT 'Marca de tiempo de última actualización'
)
USING DELTA
COMMENT 'Tabla de parámetros de control para la orquestación incremental y estado de pipelines'
""")

In [0]:
# ============================================================
# LECTURA DE LA TABLA DE CONTROL
# ============================================================

# Lee la configuración actual del pipeline desde pipeline_control
# dentro del esquema de metadata en el catálogo prod_dataco.
# Permite verificar los datos registrados antes de continuar
# con la preparación del proceso.

df_control = spark.read.table(
    "prod_dataco.metadata.pipeline_control"
)

display(df_control)

In [0]:
# ============================================================
# CONFIGURACIÓN INICIAL DEL PIPELINE
# ============================================================

# Define los valores iniciales del pipeline.
# Estos datos se utilizarán para inicializar pipeline_control
# cuando el registro del pipeline aún no exista.
from datetime import datetime

data = [
    (
        1,
        "databricks_extract_performance_logistico_dataco",
        "ingest_sharepoint_bronze",
        "incremental",
        "weekly",
        "2015-12-31",
        "2016-01-01",
        "2016-01-07",
        "pending",
        datetime.now(),
        datetime.now()
    )
]

columns = [
    "pipeline_id",
    "pipeline_name",
    "task_name",
    "load_type",
    "load_frequency",
    "last_successful_end_date",
    "next_start_date",
    "next_end_date",
    "status",
    "created_at",
    "updated_at"
]

df_initial = spark.createDataFrame(data, columns)

display(df_initial)

In [0]:
# ============================================================
# CONVERSIÓN DE CAMPOS DE FECHA
# ============================================================

# Convierte los campos de fecha de STRING a tipo DATE
# para asegurar que puedan utilizarse correctamente
# en las validaciones y cálculos del pipeline.
from pyspark.sql import functions as F

df_initial = (
    df_initial
    .withColumn(
        "last_successful_end_date",
        F.to_date("last_successful_end_date")
    )
    .withColumn(
        "next_start_date",
        F.to_date("next_start_date")
    )
    .withColumn(
        "next_end_date",
        F.to_date("next_end_date")
    )
)

df_initial.printSchema()

In [0]:
# ============================================================
# VALIDACIÓN E INICIALIZACIÓN DEL PIPELINE
# ============================================================

# Verifica si el pipeline ya existe en la tabla de control.
# Se utiliza pipeline_id como identificador único del pipeline.

pipeline_exists = (
    spark.table("prod_dataco.metadata.pipeline_control")
    .filter(F.col("pipeline_id") == 1)
    .limit(1)
    .count() > 0
)

if not pipeline_exists:
    (
        df_initial.write
        .format("delta")
        .mode("append")
        .saveAsTable("prod_dataco.metadata.pipeline_control")
    )
    print("Pipeline inicializado correctamente en prod_dataco.metadata.pipeline_control.")
else:
    print("El pipeline ya existe. No se realizará ninguna inserción.")

In [0]:
# ============================================================
# VALIDACIÓN DE LA CONFIGURACIÓN DEL PIPELINE
# ============================================================

# Lee nuevamente la tabla de control después de la inicialización.
# Permite verificar que la configuración del pipeline
# se encuentre correctamente registrada en prod_dataco.metadata.
df_control = spark.read.table(
    "prod_dataco.metadata.pipeline_control"
)

display(df_control)

In [0]:
# ============================================================
# CARGA DE LA CONFIGURACIÓN DEL PIPELINE
# ============================================================

# Lee la configuración actual del pipeline desde la tabla de control
# dentro del esquema metadata del catálogo prod_dataco.
# El DataFrame será utilizado en las siguientes etapas
# para obtener el rango de fechas a procesar.

df_control = spark.read.table(
    "prod_dataco.metadata.pipeline_control"
)

In [0]:
# ============================================================
# OBTENCIÓN DEL RANGO DE FECHAS A PROCESAR
# ============================================================

# Filtra la configuración correspondiente al pipeline actual
# y obtiene las fechas definidas para la siguiente extracción.

from pyspark.sql import functions as F

df_corte = (
    df_control
    .filter(
        F.col("pipeline_name") == "databricks_extract_performance_logistico_dataco"
    )
    .select(
        "next_start_date",
        "next_end_date"
    )
)

display(df_corte)

In [0]:

# ============================================================
# ASIGNACIÓN DEL RANGO DE FECHAS
# ============================================================

# Obtiene el único registro de corte configurado para el pipeline.
corte = df_corte.first()

fecha_inicio = corte["next_start_date"]
fecha_fin = corte["next_end_date"]

print(f"Fecha inicio de extracción: {fecha_inicio}")
print(f"Fecha fin de extracción: {fecha_fin}")

In [0]:
# ============================================================
# VALIDACIÓN DE CANTIDAD DE CORTES
# ============================================================

# Verifica que exista exactamente un corte configurado
# para el pipeline.
cantidad_cortes = df_corte.count()

if cantidad_cortes == 0:
    raise ValueError(
        "No existe un corte configurado para el pipeline."
    )

if cantidad_cortes > 1:
    raise ValueError(
        f"Se encontraron {cantidad_cortes} cortes para el pipeline. "
        "Debe existir exactamente uno."
    )

print("Validación OK: existe un único corte para el pipeline.")

In [0]:
# ============================================================
# VALIDACIÓN DEL RANGO DE FECHAS
# ============================================================

# Verifica que la fecha de inicio esté informada.
if fecha_inicio is None:
    raise ValueError(
        "El campo next_start_date está vacío."
    )

if fecha_fin is None:
    raise ValueError(
        "El campo next_end_date está vacío."
    )

if fecha_inicio > fecha_fin:
    raise ValueError(
        f"El rango de fechas es inválido: "
        f"{fecha_inicio} > {fecha_fin}"
    )

print(
    f"Validación OK: rango válido "
    f"{fecha_inicio} → {fecha_fin}"
)

In [0]:
# ============================================================
# PUBLICACIÓN DE FECHA DE INICIO
# ============================================================

# Publica la fecha de inicio como parámetro del Task
# para que pueda ser utilizada por las siguientes tareas
# del Workflow.
dbutils.jobs.taskValues.set(
    key="fecha_inicio",
    value=str(fecha_inicio)
)

In [0]:

# ============================================================
# PUBLICACIÓN DE FECHA DE FIN
# ============================================================

# Publica la fecha de fin como parámetro del Task
# para que pueda ser utilizada por las siguientes tareas
# del Workflow.
dbutils.jobs.taskValues.set(
    key="fecha_fin",
    value=str(fecha_fin)
)

In [0]:
# ============================================================
# PUBLICACIÓN DEL ID DE EJECUCIÓN
# ============================================================

# Publica el identificador único de la ejecución
# para que pueda ser utilizado por las siguientes tareas
# del Workflow.
dbutils.jobs.taskValues.set(

    key="id_ejecucion",

    value=id_ejecucion

)

In [0]:
# ============================================================
# VALIDACIÓN DE PARÁMETROS PUBLICADOS
# ============================================================

# Muestra los parámetros que fueron publicados al Workflow
# para verificar que los valores sean correctos.
print("Parámetros publicados correctamente.")
print(f"Fecha inicio: {fecha_inicio}")
print(f"Fecha fin:    {fecha_fin}")
print(f"ID ejecución: {id_ejecucion}")

In [0]:
# ============================================================
# REGISTRO DE EJECUCIÓN EN TABLA DE CONTROL
# ============================================================

from pyspark.sql import functions as F

TABLA_EJECUCION_CONTROL = "prod_dataco.metadata.ejecucion_control"

# Crear la tabla de control de ejecuciones si aún no existe en prod_dataco.metadata
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLA_EJECUCION_CONTROL} (
    id_ejecucion           STRING     COMMENT 'Identificador único de la ejecución del pipeline',
    fecha_inicio           DATE       COMMENT 'Fecha de inicio del periodo procesado',
    fecha_fin              DATE       COMMENT 'Fecha de fin del periodo procesado',
    fecha_inicio_ejecucion TIMESTAMP  COMMENT 'Marca de tiempo exacta de inicio de ejecución',
    fecha_fin_ejecucion    TIMESTAMP  COMMENT 'Marca de tiempo exacta de finalización',
    estado                 STRING     COMMENT 'Estado del pipeline (ej. RUNNING, SUCCESS, FAILED)',
    es_ejecucion_valida    BOOLEAN    COMMENT 'Indica si la ejecución finalizó correctamente'
)
USING DELTA
COMMENT 'Tabla de control y auditoría para el seguimiento de ejecuciones de pipelines'
""")

# Registrar la nueva ejecución con estado inicial EN_PROCESO
df_ejecucion = spark.createDataFrame(
    [
        (
            id_ejecucion,
            fecha_inicio,
            fecha_fin,
            None,
            None,
            "EN_PROCESO",
            False
        )
    ],
    """
    id_ejecucion STRING,
    fecha_inicio DATE,
    fecha_fin DATE,
    fecha_inicio_ejecucion TIMESTAMP,
    fecha_fin_ejecucion TIMESTAMP,
    estado STRING,
    es_ejecucion_valida BOOLEAN
    """
)

df_ejecucion = (
    df_ejecucion
    .withColumn("fecha_inicio_ejecucion", F.current_timestamp())
)

df_ejecucion.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TABLA_EJECUCION_CONTROL)

print(f"✓ Ejecución registrada en {TABLA_EJECUCION_CONTROL}: {id_ejecucion}")
print(f"✓ Corte: {fecha_inicio} → {fecha_fin}")
print("✓ Estado: EN_PROCESO")

In [0]:
# ============================================================
# VALIDACIÓN DE EJECUCIÓN REGISTRADA
# ============================================================

display(
    spark.table("prod_dataco.metadata.ejecucion_control")
    .filter(F.col("id_ejecucion") == id_ejecucion)
)

In [0]:
%sql
-- ============================================================
-- VALIDACIÓN DE LA CONFIGURACIÓN DEL PIPELINE
-- ============================================================

-- Muestra los registros configurados en pipeline_control
-- dentro del nuevo catálogo prod_dataco y esquema metadata,
-- ordenados desde el más antiguo al más reciente.

SELECT *
FROM prod_dataco.metadata.pipeline_control
ORDER BY created_at;

In [0]:
# ============================================================
# FIN DE LOG ETL - EXITOSO
# ============================================================

from delta.tables import DeltaTable

delta_log = DeltaTable.forName(
    spark,
    TABLA_ETL_LOG
)

delta_log.update(
    condition=f"log_id = '{log_id}'",
    set={
        "fecha_fin": "current_timestamp()",
        "estado": "'EXITOSO'",
        "filas_procesadas": "NULL",
        "mensaje_error": "NULL"
    }
)

print("✓ ETL Log finalizado correctamente")
print(f"✓ log_id: {log_id}")
print("✓ Estado: EXITOSO")